# Hamiltonian Simulation Case Study

## Problem

Use `quantum-backend-bench` to study how a first-order Trotterized Ising-style Hamiltonian simulation scales as the number of qubits and Trotter steps change.

The benchmark implements:

`H = sum_i Z_i Z_{i+1} + 0.5 * sum_i X_i`

## Quantum Advantage

None is claimed in this notebook. These are small local simulator runs intended to make benchmark structure, runtime, and reproducibility visible. The same workflow can be scaled cautiously for larger local studies.

## SDK Advantages

- Cirq gives a fast, credential-free first path for local circuit simulation.
- Qiskit Aer is useful when users want a common local simulator workflow that includes transpilation-style overhead in this package's adapter timing.

## Variables and Parameters

- `case_grid`: Hamiltonian simulation cases to run.
- `active_backends`: installed local backends used for the case study.
- `shots`: measurement samples per run.
- `repeats`: repeated executions for runtime samples.
- `time`: evolution time parameter.
- `trotter_steps`: first-order Trotter step count.


## Setup

Import the public package APIs and create the shared notebook artifact directory.


In [ ]:
import importlib.util

import matplotlib.pyplot as plt

from quantum_backend_bench.benchmarks.hamiltonian_sim import build_benchmark
from quantum_backend_bench.core.runner import run_benchmark
from quantum_backend_bench import results_to_dataframe
from quantum_backend_bench.utils.formatting import format_results_table
from quantum_backend_bench.utils.notebook import (
    check_runtime_samples,
    check_total_counts,
    notebook_artifact_dir,
    save_result_artifacts,
    top_measurement_states_frame,
    verification_frame,
)

ARTIFACT_DIR = notebook_artifact_dir()

candidate_backends = {
    "cirq": ["cirq"],
    "qiskit_aer": ["qiskit", "qiskit_aer"],
}
active_backends = [
    backend
    for backend, modules in candidate_backends.items()
    if all(importlib.util.find_spec(module) is not None for module in modules)
]

shots = 256
repeats = 2
case_grid = [
    {"n_qubits": 3, "time": 0.5, "trotter_steps": 1},
    {"n_qubits": 3, "time": 0.5, "trotter_steps": 2},
    {"n_qubits": 4, "time": 0.5, "trotter_steps": 1},
    {"n_qubits": 4, "time": 1.0, "trotter_steps": 2},
]

print("Active local backends:", ", ".join(active_backends) if active_backends else "none")

## Run the Case Grid

Each case uses the same package benchmark builder. The result dictionaries include runtime samples, structural metrics, measurement counts, and backend caveats.


In [ ]:
if not active_backends:
    raise RuntimeError(
        "No local execution backend is installed. Install at least quantum-backend-bench[cirq]."
    )

results = []
for case in case_grid:
    benchmark = build_benchmark(**case)
    results.extend(run_benchmark(benchmark, active_backends, shots=shots, repeats=repeats))

print(format_results_table(results))

frame = results_to_dataframe(results)
columns = [
    "case_label",
    "backend",
    "n_qubits",
    "runtime_seconds",
    "runtime_seconds_stddev",
    "depth",
    "gate_count",
    "two_qubit_gate_count",
]
display_frame = frame[columns].copy()
for column in ["runtime_seconds", "runtime_seconds_stddev"]:
    display_frame[column] = display_frame[column].round(6)
display_frame

## Runtime Scaling

Runtime is local-machine dependent, but this view helps users see how each SDK adapter responds to larger Trotterized circuits in the same environment.


In [ ]:
plot_frame = display_frame.copy()
plot_frame["case"] = plot_frame["case_label"].str.replace("hamiltonian_sim ", "", regex=False)

fig, ax = plt.subplots(figsize=(9, 4))
for backend, group in plot_frame.groupby("backend"):
    ax.plot(group["case"], group["runtime_seconds"], marker="o", label=backend)
ax.set_title("Hamiltonian simulation runtime by case")
ax.set_xlabel("case")
ax.set_ylabel("seconds")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="backend")
plt.tight_layout()
plt.show()

## Circuit Structure

Depth and gate counts reflect the Trotterized circuit construction. Increasing `trotter_steps` should increase circuit depth and gate count.


In [ ]:
structure = (
    frame[["case_label", "n_qubits", "depth", "gate_count", "two_qubit_gate_count"]]
    .drop_duplicates()
    .sort_values(["n_qubits", "case_label"])
)
structure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
structure.plot(x="case_label", y="depth", kind="bar", ax=axes[0], color="#e76f51", legend=False)
axes[0].set_title("Circuit depth")
axes[0].set_ylabel("depth")
axes[0].tick_params(axis="x", rotation=30)
structure.plot(
    x="case_label", y="gate_count", kind="bar", ax=axes[1], color="#457b9d", legend=False
)
axes[1].set_title("Gate count")
axes[1].set_ylabel("gates")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Example Output Distribution

The next table shows the top measured states for one selected case/backend. States are ordered numerically and displayed in ket notation.


In [ ]:
selected = results[0]
state_table = top_measurement_states_frame(selected, top_k=8, sort_by_state=True)
print(f"Selected case: {selected['metadata']['case_label']} / {selected['backend']}")
state_table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(state_table["state"], state_table["probability"], color="#2a9d8f")
ax.set_title("Top measured states for selected Hamiltonian case")
ax.set_xlabel("state")
ax.set_ylabel("probability")
ax.set_ylim(0, max(0.05, state_table["probability"].max() * 1.2))
plt.tight_layout()
plt.show()

## Save Reproducible Artifacts


In [ ]:
json_path, csv_path = save_result_artifacts(results, "hamiltonian_case_study", ARTIFACT_DIR)
print(f"Saved JSON: {json_path}")
print(f"Saved CSV: {csv_path}")

## Verification

These checks confirm that every run produced the expected total shot count, each result has the requested runtime samples, and increasing Trotter steps increased structural cost for the three-qubit cases.


In [ ]:
checks = []
for result in results:
    total_check = check_total_counts(result, expected=shots * repeats)
    total_check["check"] = f"{result['metadata']['case_label']} / {result['backend']} total shots"
    checks.append(total_check)

    runtime_check = check_runtime_samples(result, expected=repeats)
    runtime_check["check"] = (
        f"{result['metadata']['case_label']} / {result['backend']} runtime samples"
    )
    checks.append(runtime_check)

three_qubit = structure[structure["n_qubits"] == 3].sort_values("depth")
checks.append(
    {
        "check": "3-qubit depth increases with more Trotter steps",
        "value": " < ".join(str(value) for value in three_qubit["depth"]),
        "expected": "monotonic increase",
        "passed": three_qubit["depth"].is_monotonic_increasing
        and three_qubit["depth"].nunique() > 1,
    }
)

verification = verification_frame(checks)
verification